In [ ]:
import tempfile
import os
from ansys.aedt.core import Desktop, generate_unique_name
import optuna
import script_otim as script
import logging
import sys
import numpy as np

In [ ]:
optuna.logging.get_logger("optuna").addHandler(logging.StreamHandler(sys.stdout))
study_name = "Optimization_weight_param" 
storage_name = "sqlite:///{}.db".format(study_name)

In [ ]:
tmpfold = tempfile.gettempdir()

temp_folder = os.path.join(
    tmpfold,
    generate_unique_name("Antenna")
)

if not os.path.exists(temp_folder):
    os.mkdir(temp_folder)

non_graphical = os.getenv(
    "PYAEDT_NON_GRAPHICAL",
    "True"
).lower() in ("true", "1", "t")

d = Desktop(
    "2025.1",
    non_graphical=non_graphical,
    new_desktop=True
)


In [ ]:
def objective(trial):
    a1 = trial.suggest_float("a1", 1.0, 1.0)
    a2 = trial.suggest_float("a2", 1.0, 1.0)
    n1 = trial.suggest_int("n1", 1, 100)
    n2 = trial.suggest_float("n2", 1.0, 10.0)
    n3 = trial.suggest_float("n3", 1.0, 10.0)
    n4 = trial.suggest_int("n4", 1, 100)
    perc = trial.suggest_float("perc", 0.2, 0.8)
    r = trial.suggest_int("r", 90, 150)

    df = script.makeAntenna(a1, a2, n1, n2, n3, n4, perc, r, temp_folder)
    s11 = np.asarray(df['|$S_{11}$| [dB]'], dtype=float)

    n = len(s11)
    idx1 = n // 3
    idx2 = 2 * n // 3

    weight = np.zeros(n)
    weight[:idx1] = 1
    weight[idx1:idx2] = 0.9
    weight[idx2:] = 0.7
    
    mask = s11 >= -10
    
    weighted_count = np.sum(mask * weight)

    weighted_fraction = weighted_count / np.sum(weight)


    return weighted_fraction

study = optuna.create_study(
    directions=["minimize"],    
    study_name=study_name, 
    storage=storage_name, 
    load_if_exists=True
)



In [ ]:
study.optimize(objective, n_trials=1000)

In [ ]:
len(study.trials)

In [ ]:
study.best_params

In [ ]:
study.best_trial

In [ ]:
d.release_desktop()